# Stage 3 — Decision: LangGraph Walkthrough

**Chapters 7–8 | Framework: LangGraph**
**Author: Pushparajan Ramar**

This notebook walks through the Decision-stage conversion graph step by step:

1. **State definition** — `DecisionJourneyState`
2. **Tool exploration** — objection handling, battlecards, close offers, CRM
3. **Routing logic** — conditional edge decisions
4. **Graph assembly** — building and compiling the StateGraph
5. **Running scenarios** — three deal archetypes end to end

In [ ]:
# File      : decision_graph_walkthrough.ipynb
# Stage     : 3 — Decision
# Chapter   : 7–8
# Framework : LangGraph
# Author    : Pushparajan Ramar
# Repo      : https://github.com/Pushparajan/agenticai-marketing

import os, sys

# Ensure mock mode and repo root on path
os.environ["USE_MOCK"] = "true"
_repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f"Repo root: {_repo_root}")
print(f"USE_MOCK : {os.environ['USE_MOCK']}")

## 1. State Definition

The `DecisionJourneyState` is a `TypedDict` that flows through every node in the graph. It tracks deal metadata, buyer signals, objections, competitors, and the message log.

In [ ]:
from stage3_decision.graph.state import DecisionJourneyState
import typing

# Show all fields and their types
print("DecisionJourneyState fields:\n")
hints = typing.get_type_hints(DecisionJourneyState, include_extras=True)
for field, ftype in hints.items():
    print(f"  {field:25s} : {ftype}")

## 2. Exploring the Tools

Each tool has a `USE_MOCK` fallback. Let's try them individually.

### 2a. Objection Handler Tools

In [ ]:
from stage3_decision.tools.objection_handler_tools import (
    get_common_objections,
    generate_objection_response,
)

# Fetch common objections for fintech
print("=== Common Objections (fintech) ===")
print(get_common_objections.invoke({"industry": "fintech"}))

print("\n=== Objection Response ===")
print(generate_objection_response.invoke({
    "objection": "Too expensive compared to alternatives",
    "product": "MarketingAI Platform",
}))

### 2b. Competitive Battlecard Tools

In [ ]:
from stage3_decision.tools.competitive_battlecard_tools import (
    get_battlecard,
    get_competitive_comparison,
)

print("=== Battlecard: CompetitorX ===")
print(get_battlecard.invoke({"competitor": "CompetitorX"}))

print("\n=== Comparison: RivalY on security ===")
print(get_competitive_comparison.invoke({
    "competitor": "RivalY",
    "feature_area": "security",
}))

### 2c. Close Offer Tools

In [ ]:
from stage3_decision.tools.close_offer_tools import (
    generate_close_offer,
    create_urgency_campaign,
)

print("=== Close Offer ($75k, enterprise) ===")
print(generate_close_offer.invoke({"deal_value": 75000.0, "tier": "enterprise"}))

print("\n=== Urgency Campaign (18 days stalled) ===")
print(create_urgency_campaign.invoke({
    "contact_id": "contact-303",
    "days_stalled": 18,
}))

### 2d. Deal CRM Tools

In [ ]:
from stage3_decision.tools.deal_crm_tools import (
    get_deal,
    update_deal_stage,
    create_deal_task,
)

print("=== Get Deal ===")
print(get_deal.invoke({"deal_id": "deal-001"}))

print("\n=== Update Deal Stage ===")
print(update_deal_stage.invoke({"deal_id": "deal-001", "stage": "negotiation"}))

print("\n=== Create Task ===")
print(create_deal_task.invoke({
    "deal_id": "deal-001",
    "title": "Follow up on pricing discussion",
    "owner": "sales-rep-a",
}))

## 3. Routing Logic

The `route_decision_action` function inspects state and picks the next node. Let's test it with different signal combinations.

In [ ]:
from stage3_decision.graph.routing import route_decision_action

test_cases = [
    {
        "label": "High-value deal, not approved",
        "state": {"deal_value": 60000, "human_approved": False, "intent_score": 90,
                  "proposal_viewed": True, "objections_raised": [], "competitors_named": [],
                  "days_in_decision": 5},
        "expected": "request_human_approval",
    },
    {
        "label": "High intent + proposal viewed (under $50k)",
        "state": {"deal_value": 40000, "human_approved": False, "intent_score": 90,
                  "proposal_viewed": True, "objections_raised": [], "competitors_named": [],
                  "days_in_decision": 5},
        "expected": "send_close_offer",
    },
    {
        "label": "Objections raised",
        "state": {"deal_value": 30000, "human_approved": False, "intent_score": 60,
                  "proposal_viewed": False, "objections_raised": ["Too expensive"],
                  "competitors_named": [], "days_in_decision": 5},
        "expected": "address_objections",
    },
    {
        "label": "Competitor named",
        "state": {"deal_value": 30000, "human_approved": False, "intent_score": 60,
                  "proposal_viewed": False, "objections_raised": [],
                  "competitors_named": ["RivalY"], "days_in_decision": 5},
        "expected": "send_comparison_content",
    },
    {
        "label": "Stalled > 14 days",
        "state": {"deal_value": 30000, "human_approved": False, "intent_score": 40,
                  "proposal_viewed": False, "objections_raised": [], "competitors_named": [],
                  "days_in_decision": 18},
        "expected": "reactivate_with_urgency",
    },
    {
        "label": "Default nurture",
        "state": {"deal_value": 30000, "human_approved": False, "intent_score": 50,
                  "proposal_viewed": False, "objections_raised": [], "competitors_named": [],
                  "days_in_decision": 5},
        "expected": "send_next_nurture",
    },
]

print(f"{'Label':<45} {'Result':<30} {'Pass?'}")
print("-" * 85)
for tc in test_cases:
    result = route_decision_action(tc["state"])
    passed = "YES" if result == tc["expected"] else f"NO (expected {tc['expected']})"
    print(f"{tc['label']:<45} {result:<30} {passed}")

## 4. Building the Graph

We assemble the `StateGraph`, wire up nodes and conditional edges, attach the `MemorySaver` checkpointer, and set `interrupt_before` on the human review node.

In [ ]:
from stage3_decision.graph.graph import build_decision_graph

graph = build_decision_graph()
print("Graph compiled successfully!")
print(f"  Nodes: {list(graph.nodes.keys())}")

## 5. Running Scenarios

Let's run each of the three deal archetypes through the graph.